# ⚛️ QDIT: Quantum Transfer Learning for Diabetic Retinopathy Detection
### Implementation of arXiv:2405.01734v1 on Kaggle

This notebook trains the **Hybrid Classical-Quantum Transfer Learning (CQ-TL)** model on retinal fundus images from Kaggle dataset `bhavyasanghavi2348/data-qdit` using PennyLane and PyTorch.

**Architecture**:
- **Classical Feature Extractor (A')**: Pre-trained ResNet-18 (frozen weights) mapping fundus images to 512 abstract features.
- **Dressed Quantum Circuit (B)**: Dense layer $\to$ $\tanh$ $\to$ $\frac{\pi}{2}$ scale $\to$ 4-Qubit Variational Quantum Circuit $\to$ Dense classification layer for 5 DR severity stages.

In [ ]:
# Step 1: Clone repository & install dependencies
!git clone https://github.com/Bhavya-2128/QDIT.git
%cd QDIT
# Only install missing packages - do NOT reinstall torchvision/torch to preserve CUDA kernels
!pip install -q pennylane kagglehub


In [ ]:
# Step 2: Download or locate dataset on Kaggle
import kagglehub
import os

dataset_path = kagglehub.dataset_download("bhavyasanghavi2348/data-qdit")
print("✅ Dataset path:", dataset_path)
print("Directory contents:", os.listdir(dataset_path))

In [ ]:
# Step 3: Draw the 4-Qubit Variational Quantum Circuit
from quantum.circuits import draw_circuit
from quantum.config import QuantumCircuitConfig, EmbeddingGateType, EntanglingGateType

circuit_cfg = QuantumCircuitConfig(
    n_qubits=4,
    q_depth=4,
    embedding_gate=EmbeddingGateType.HADAMARD,
    entangling_gate=EntanglingGateType.CNOT
)
print(draw_circuit(circuit_cfg))

In [ ]:
# Step 4: Train the Quantum Model
import torch
from quantum.config import ModelConfig, TrainingConfig, BackboneType
from quantum.dataset import get_dataloaders
from quantum.models import QuantumTransferLearningDR
from quantum.train import train_model
from quantum.evaluate import format_metrics_table

# Configurations
model_cfg = ModelConfig(
    backbone=BackboneType.RESNET18,
    quantum_circuit=circuit_cfg
)
train_cfg = TrainingConfig(
    batch_size=16,
    epochs=30,
    learning_rate=1e-3,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# DataLoaders with Ben Graham Gaussian Blur preprocessing
train_loader, val_loader = get_dataloaders(
    dataset_dir=dataset_path,
    batch_size=train_cfg.batch_size,
    image_size=(224, 224),
    apply_graham=True
)

# Instantiate and train
model = QuantumTransferLearningDR(config=model_cfg)
trained_model, history, best_metrics = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=train_cfg,
    save_path="/kaggle/working/quantum_dr_model.pt"
)

print(format_metrics_table(best_metrics))

In [ ]:
# Step 5: Plot Training Loss & Validation Accuracy Curves
import matplotlib.pyplot as plt

epochs_range = range(1, len(history["train_loss"]) + 1)
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], 'b-o', label='Train Loss')
plt.plot(epochs_range, history["val_loss"], 'r-s', label='Val Loss')
plt.title('Quantum Model Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross Entropy Loss')
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, [a * 100 for a in history["train_acc"]], 'b-o', label='Train Accuracy')
plt.plot(epochs_range, [a * 100 for a in history["val_acc"]], 'g-s', label='Val Accuracy')
plt.title('Classification Accuracy (%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Step 6: Test Single Image Inference & Quantum State Analysis
from quantum.infer import QuantumDRPredictor
from quantum.dataset import generate_synthetic_fundus_image

predictor = QuantumDRPredictor(checkpoint_path="/kaggle/working/quantum_dr_model.pt")
sample_scan = generate_synthetic_fundus_image(stage=2)
result = predictor.predict(sample_scan)

print(f"🩺 Diagnosis: Stage {result['predicted_stage']} ({result['stage_name']})")
print(f"🎯 Confidence: {result['confidence']*100:.2f}%")
print("⚛️  Pauli-Z Expectations:", result['quantum_pauli_z_expectations'])
print("📋 Recommendation:", result['clinical_diagnosis'])